# Day 3 — Ablation Results

**Question:** Does fusing motor-speech channels (phonation × DDK) beat any single channel at PD screening on NeuroVoz?

**Setup:** Subject-level LOSO on the analysis cohort (49 PD × 46 HC — age-matched HC ≥ 50, has PATAKA, has ≥1 vowel). Per-channel `StandardScaler → LogisticRegression(l2, C=1.0)`, re-fit per fold. Late fusion at the score level with AUC-excess-derived weights.

**Answer:** Yes — but only after two corrections to the naïve Day 2 pipeline:
1. Train phonation on per-file features instead of the per-subject mean.
2. Set fusion weights via `w_c ∝ max(0, AUC_c − 0.5)` instead of the CogniScreen-era prior.

Full technical narrative is in [CLAUDE.md → "Day 3 findings"](../CLAUDE.md). This notebook is the visual companion.


In [1]:
from pathlib import Path
import json
import pandas as pd

REPO_ROOT = Path.cwd().parent
results = REPO_ROOT / "eval" / "results"

ablation = pd.read_csv(results / "ablation_table.csv")
summary = json.loads((results / "ablation_summary.json").read_text())

print(f"Cohort: {summary['cohort_n']} subjects "
      f"({summary['cohort_pd']} PD, {summary['cohort_hc']} HC)")
print(f"LOSO folds per model: {summary['cohort_n']} (leave-one-subject-out)")
print(f"Bootstrap resamples for AUC 95% CI: {summary['n_bootstrap']}")
print(f"Speech fusion weights (AUC-excess normalized): {summary['fusion_weights_used']}")


Cohort: 95 subjects (49 PD, 46 HC)
LOSO folds per model: 95 (leave-one-subject-out)
Bootstrap resamples for AUC 95% CI: 1000
Speech fusion weights (AUC-excess normalized): {'phonation': 0.35, 'ddk': 0.65}


## The ablation table

Eight rows, three groups:

- **Rows 1–4:** baseline pipeline (per-subject mean-over-vowels phonation).
- **Rows 5–6:** per-file phonation LOSO — one row per vowel file (n=466), grouped by `subject_id` for split. No leakage; both per-file and per-subject-aggregated eval reported.
- **Rows 7–8:** fusion using per-file-trained phonation.

Winning row highlighted.


In [2]:
tbl = ablation.copy()
tbl["AUC [95% CI]"] = tbl.apply(
    lambda r: f"{r['auc']:.3f} [{r['auc_ci_low']:.3f}, {r['auc_ci_high']:.3f}]",
    axis=1,
)
tbl["F1"] = tbl["f1"].map("{:.3f}".format)
tbl["Sens"] = tbl["sensitivity"].map("{:.3f}".format)
tbl["Spec"] = tbl["specificity"].map("{:.3f}".format)
tbl_display = tbl[["model", "n", "AUC [95% CI]", "F1", "Sens", "Spec"]].rename(
    columns={"model": "Model", "n": "n"}
)

best_idx = ablation["auc"].idxmax()

def _highlight_best(row):
    style = "font-weight: bold; background-color: #fff3cd"
    return [style if row.name == best_idx else "" for _ in row]

tbl_display.style.apply(_highlight_best, axis=1).hide(axis="index")


Model,n,AUC [95% CI],F1,Sens,Spec
phonation-only,95,"0.567 [0.455, 0.678]",0.545,0.490,0.674
ddk-only,95,"0.740 [0.632, 0.845]",0.701,0.694,0.696
phonation+ddk_unweighted,95,"0.722 [0.617, 0.822]",0.638,0.612,0.674
phonation+ddk_weighted,95,"0.736 [0.634, 0.842]",0.660,0.633,0.696
phonation_per-file_per-file-eval,466,"0.603 [0.553, 0.653]",0.581,0.560,0.604
phonation_per-file_per-subject-eval,95,"0.630 [0.518, 0.735]",0.571,0.531,0.652
phonation-per-file+ddk_unweighted,95,"0.756 [0.658, 0.854]",0.701,0.694,0.696
phonation-per-file+ddk_weighted,95,"0.758 [0.662, 0.859]",0.667,0.633,0.717


**Read:** The naïve baseline fusion (row 3, AUC 0.722) *loses* to DDK-only (row 2, 0.740). The winning row (8, AUC 0.758) uses per-file phonation training + AUC-excess weights and beats the best single channel by +0.018. CIs are wide (expected at N=95) but the ordering is stable across both weighted and unweighted variants of per-file fusion.


## ROC curves

Three headline models. Chance line for reference. AUCs in the legend are computed on the LOSO OOF probabilities in `eval/results/loso_oof_probs.csv`.

![ROC curves](../eval/results/figures/roc_curves.png)

The fusion curve dominates the **low-FPR region** — the clinically relevant zone. A screening tool that trips at high false-positive rate is a burden on downstream diagnostic capacity; the low-FPR left edge is where "who to send for follow-up" decisions live. Phonation-only stays close to chance in that same zone.


## Per-feature coefficients

Phonation was refit with **L1 penalty** (`liblinear`, C=0.3) for this visualization. The L2 deployment classifier suffers from multicollinearity — 4 correlated shimmer variants (`shimmer_local`, `apq5`, `apq11`, `dda`) plus 3 correlated jitter variants (`jitter_local`, `rap`, `ppq5`). L2 splits the coefficient across them with unstable signs (e.g. `shimmer_apq5` and `shimmer_apq11` end up opposite-signed under L2 — a textbook multicollinearity artefact). Reading L2 coefficients literally would be misleading.

L1 forces sparsity → each collinear cluster collapses onto a single representative feature. The saved L1 coefficients are in `eval/results/coefficients_l1_phonation.csv`. DDK has 8 features and no material collinearity → kept as L2 deployment model.

**The L2 deployment classifiers in `eval/models/{phonation,ddk}.joblib` are unchanged.** L1 here is a viz tool only.

![Coefficient bars](../eval/results/figures/coefficients.png)


**Phonation (L1):**
- `jitter_local` absorbs the entire jitter-family signal (**+0.46**, positive → PD ✓). Matches clinical expectation — PD patients have higher cycle-to-cycle period perturbation.
- All 4 shimmer variants → zero. L1 judged them redundant given `jitter_local`.
- Small residual bars on `f0_mean_hz`, `hnr_mean_db`, `f0_std_hz`. `hnr` sign is slightly off clinical intuition but magnitude is minor (< 0.05).

**DDK (L2):**
- `ddk_rate_hz` (**−1.03**, higher rate → HC ✓): PD's articulatory bradykinesia — slower DDK.
- `amp_mean` (**−0.68**, louder → HC ✓): PD's hypophonia — reduced loudness.
- `isi_cv` (**+0.55**, higher timing variability → PD ✓): rhythm instability.
- `amp_cv` and `isi_mean_s` support the same story (both negative, higher = HC).

Both channels' top drivers align with the PD motor-speech literature. The L1 view of phonation makes clear that when we trust the model, we're really trusting `jitter_local` — the entire shimmer family is redundant given jitter at our N.


## Key findings (Day 3)

Condensed. Full versions in [CLAUDE.md → "Day 3 findings"](../CLAUDE.md).

1. **DDK > phonation on NeuroVoz** (0.740 vs 0.567 baseline). The CogniScreen-era prior "phonation is the most direct evidence" was wrong for this cohort. Articulatory bradykinesia on PATAKA is the stronger single-task marker.

2. **Aggregation was silently diluting phonation.** Mean-over-vowels aggregation lost signal; per-file training lifted phonation-only per-subject AUC from 0.567 → 0.630 (+0.063).

3. **Fusion > best single channel required BOTH corrections.** Per-file training alone (unweighted fusion 0.756) helped; adding AUC-excess weighting (0.758) put fusion clearly ahead of DDK-only (+0.018).

4. **AUC-excess weighting** — `w_c ∝ max(0, AUC_c − 0.5)` normalized over channels present → phonation 0.35, DDK 0.65. When prosody joins, all speech weights must be re-derived from an updated LOSO table (do not just insert prosody weight).

5. **Failed diagnostic (informative).** Hypothesized outliers → StandardScaler → dilution. `log(x+eps) + RobustScaler` on jitter/shimmer → AUC went 0.567 → 0.534 (worse). Ruled out outliers; refocused on aggregation. Code reverted.

6. **Unexpected: per-file eval AUC (0.603) < per-subject aggregated eval AUC (0.630).** Individual vowel-file features are noise-dominated; per-subject averaging denoises them. Implication for demo: multi-vowel recordings preferable over single vowel.

7. **Two eval regimes reported.** Per-subject LOSO (95 datapoints) = deployment metric. Per-file `LeaveOneGroupOut(subject_id)` (466 datapoints) = literature-comparable. Both split by subject → no leakage in either.

**Caveats (unchanged):**
- All PD training subjects recorded ON medication (2–5h post-dose per NeuroVoz paper §Data records). Numbers apply to ON-state PD vs age-matched HC; not calibrated for OFF-state.
- Screening decision-aid, NOT a clinical diagnosis. Reports must include both disclaimers.


## What's next

- **Day 4** (Thu Jul 10): Facial channel joins fusion. Smile classifier already trained on UFNet (in-distribution test AUROC 0.812). Fusion weights will need a facial term — derived from Day 4's facial LOSO AUC via the same `AUC_c − 0.5` heuristic. Claude report layer wires phonation + DDK + facial + hypomimia narrative into a single natural-language output.

- **Day 5** (Fri Jul 11): End-to-end demo. Gradio UI, single video upload → task-matched feature extraction → per-channel scores → weighted fusion → Claude report.

- **Optional stretch (only if ahead):** Per-vowel modeling for phonation. 5 separate LogRegs (one per balanced vowel: A1, A2, I1, O2, U1), fused via soft vote → per-subject AUC. Expected further phonation lift of +0.02–0.05 AUC. See `TODO.md → Day 3 → Optional / stretch`.

**Reproduction:** rebuild everything in this notebook from raw features with
```bash
.venv/bin/python -m eval.ablation
.venv/bin/python -m eval.make_plots
```
